<a href="https://colab.research.google.com/github/ShikharV010/gist_daily_runs/blob/main/BacklinkContentAutomaton.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Step 0: Install dependencies & authenticate Google

In [ ]:
### Uncomment the following segment for manual runs!

# !pip install anthropic google-auth google-auth-oauthlib google-api-python-client gspread -q

# from google.colab import auth
# auth.authenticate_user()
# print('✓ Google authenticated')


### For automated runs
import os
import json
from google.oauth2.service_account import Credentials

SCOPES = [
    'https://www.googleapis.com/auth/documents',
    'https://www.googleapis.com/auth/drive',
    'https://www.googleapis.com/auth/spreadsheets'
]

service_account_info = json.loads(os.environ['GOOGLE_SERVICE_ACCOUNT_JSON'])
creds = Credentials.from_service_account_info(service_account_info, scopes=SCOPES)
print('✓ Google authenticated via service account')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 14.8 MB/s eta 0:00:00
✓ Google authenticated


#Step 1: Config Cell

In [ ]:
# === CONFIG ===

ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')       # <-- Paste your Claude API key here
MODEL = 'claude-sonnet-4-6'  # Updated to latest model
API_DELAY = 2                # seconds between Claude calls

# Work Order Sheet (your input sheet)
WO_SHEET_URL = 'https://docs.google.com/spreadsheets/d/1ceFYnAmzTZDBCn4F28KGKiMEu-4QD5maYgtEuY6iDek/edit?usp=sharing'

# Tracker Sheet (progress + metadata tracking)
# First run  → leave empty, a new sheet is created and ID is printed
# On restart → paste the ID printed during first run
TRACKER_SHEET_ID = '1z_daQxBo7WAQNRLCp8CAWJz7ZzaeHbZnWbnlpJgQk3M'        # <-- Paste tracker sheet ID here on restart

print('✓ Config set')
if not ANTHROPIC_API_KEY:
    print('⚠ PASTE YOUR ANTHROPIC API KEY above!')

✓ Config set


#Step 2: Initialize APIs & tracker sheet

In [ ]:
import anthropic
import google.auth
import gspread
from googleapiclient.discovery import build
import json, time, re, pandas as pd
from datetime import datetime

# ─── Clients ──────────────────────────────────────────────────────────────────
claude = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
docs_service = build('docs', 'v1', credentials=creds)
drive_service = build('drive', 'v3', credentials=creds)
gc = gspread.authorize(creds)

In [ ]:
# ─── Work Order Sheet ─────────────────────────────────────────────────────────
wo_sheet = gc.open_by_url(WO_SHEET_URL)
wo_ws = wo_sheet.sheet1

# Load into DataFrame, preserving actual sheet row numbers for write-back
wo_data = wo_ws.get_all_records()
wo_df = pd.DataFrame(wo_data)
wo_df.columns = wo_df.columns.str.strip()
wo_df['wo_key'] = (
    wo_df['Domain'].str.strip() + '||' +
    wo_df['Page'].str.strip() + '||' +
    wo_df['BL Website'].str.strip()
)
wo_df['_sheet_row'] = range(2, len(wo_df) + 2)  # row 1 = header, data starts at 2

# Ensure "Content Link" column exists in WO sheet
# Ensure "Content Link" column exists in WO sheet
wo_headers = wo_ws.row_values(1)
if 'Content Link' not in wo_headers:
    wo_ws.update_cell(1, len(wo_headers) + 1, 'Content Link')
    wo_headers.append('Content Link')
content_link_col = wo_headers.index('Content Link') + 1  # 1-indexed for gspread
status_col = wo_headers.index('Status') + 1              # ← add this line

print(f'✓ Connected to WO sheet ({len(wo_df)} work orders)')
print(f'  "Content Link" → column {content_link_col}')
print(f'  "Status" → column {status_col}')

✓ Connected to WO sheet (594 work orders)
  "Content Link" → column 7
  "Status" → column 8


In [ ]:
# ─── Tracker Sheet ────────────────────────────────────────────────────────────
TRACKER_HEADERS = [
    'WO Key', 'Domain', 'Page', 'Keyword', 'Anchor Text',
    'BL Website', 'Topic', 'Hook Type', 'Word Count',
    'Google Doc URL', 'Status', 'Processed At'
]

if not TRACKER_SHEET_ID:
    raise ValueError("⚠ TRACKER_SHEET_ID is empty! Paste your tracker sheet ID in the config cell.")

tracker_sheet = gc.open_by_key(TRACKER_SHEET_ID)
tracker_ws = tracker_sheet.sheet1

# Ensure headers exist (handles both fresh and existing sheets)
if not tracker_ws.row_values(1):
    tracker_ws.append_row(TRACKER_HEADERS, value_input_option='RAW')
    print('✓ Connected to tracker sheet (headers added — fresh sheet)')
else:
    existing_row_count = len(tracker_ws.get_all_values()) - 1
    print(f'✓ Connected to tracker sheet ({existing_row_count} rows already logged)')

tracker_url = f'https://docs.google.com/spreadsheets/d/{TRACKER_SHEET_ID}/edit'
print(f'\n📊 Tracker URL: {tracker_url}')
print('✓ All APIs ready')

✓ Connected to tracker sheet (23 rows already logged)

📊 Tracker URL: https://docs.google.com/spreadsheets/d/1z_daQxBo7WAQNRLCp8CAWJz7ZzaeHbZnWbnlpJgQk3M/edit
✓ All APIs ready


In [ ]:
# ─── Detect already-completed WOs (skip on re-run) ───────────────────────────

# Method 1: Already logged in tracker sheet with Status = Success
existing_records = tracker_ws.get_all_records()
completed_keys_tracker = {
    r['WO Key'] for r in existing_records if r.get('Status') == 'Success'
}

# Method 2: Already has a Content Link in the WO sheet itself
completed_keys_wo = set(
    wo_df[wo_df['Content Link'].str.strip().str.len() > 0]['wo_key']
) if 'Content Link' in wo_df.columns else set()

# Combine both
completed_keys = completed_keys_tracker | completed_keys_wo

pending_df = wo_df[~wo_df['wo_key'].isin(completed_keys)].reset_index(drop=True)

print(f'\nTotal work orders:         {len(wo_df)}')
print(f'Already completed (WO):    {len(completed_keys_wo)}')
print(f'Already logged (tracker):  {len(completed_keys_tracker)}')
print(f'Remaining to process:      {len(pending_df)}')

if len(pending_df) == 0:
    print('\n🎉 All work orders already processed!')


Total work orders:         594
Already completed (WO):    585
Already logged (tracker):  21
Remaining to process:      7


#Step 3: Define pipeline functions

In [ ]:
# ==============================================================
# A: Generate 10 topic ideas
# ==============================================================

def generate_topics(domain, page_url, primary_kw, anchor_text, bl_website):

    prompt = f"""You are a senior SEO strategist and content planner tasked with generating high-impact backlink article topics for a client.

GOAL: Create 10 original article titles designed specifically for backlinking that:
- Are highly relevant to the client's product, brand, or community
- Are sharply aligned with the ICP (ideal customer persona)
- Include attention-grabbing hooks (e.g., surprising insights, bold statements, comparisons, lists)
- Match real keyword intents and organic search behaviour
- Are optimised to earn backlinks from high quality domains
- Help improve rankings for the brand name or other key SEO targets
- Include at least 5-10 listicles or comparison-based titles
- Avoid generic content or shallow information that adds no SEO or brand value

CONTEXT:
Client: {domain}
Product/Services: (infer from the client domain and page URL)
Top Keywords to Rank For: {primary_kw}
SEO Goal: Improve rankings for "{primary_kw}" through backlinks
Target Media/website for Backlinking: {bl_website}
Geo Focus: US
Page URL (for backlink): {page_url}
Anchor Text: {anchor_text}

INSTRUCTIONS:
- Create unique article titles that appeal to the target ICP even if they don't know the brand yet.
- Use formats known to earn backlinks: Lists (Top 5, 7, 10), How-to guides, Contrarian or anti-playbook insights, Comparison pieces (X vs Y), Frameworks, myths, lessons, or templates
- Include at least 3 article titles that mention the client in a natural, value-driven way.
- Ensure at least 2-3 geo-specific titles (US focus).
- Avoid fluff, buzzwords, overused formats, or low-depth topics.

OUTPUT: Return ONLY a JSON array with exactly 10 objects, each having:
- "Topic Title": string
- "Hook Type": string
- "SEO Keyword Intent": string
- "Why It Works": string

No other text outside the JSON array."""

    resp = claude.messages.create(model=MODEL, max_tokens=4000,
                                  messages=[{"role": "user", "content": prompt}])
    text = resp.content[0].text.strip()
    m = re.search(r'\[.*\]', text, re.DOTALL)
    return json.loads(m.group()) if m else json.loads(text)


# ==============================================================
# B: Auto-select best topic
# ==============================================================

def select_best_topic(topics, page_url, primary_kw, anchor_text, bl_website):

    topics_json = json.dumps(topics, indent=2)

    prompt = f"""You are a senior SEO content strategist. From the 10 article topics below, select the ONE best topic for a backlink article.

Context:
- Target page URL: {page_url}
- Primary keyword: {primary_kw}
- Anchor text: {anchor_text}
- Publishing website: {bl_website}

Selection criteria (priority order):
1. The topic must allow the anchor text "{anchor_text}" to be inserted naturally in the first H2 section
2. Strong relevance to the primary keyword "{primary_kw}"
3. High editorial value — reads like a real guest post, not promotional
4. Hook strength — would attract clicks and earn links
5. ICP alignment — useful for actual decision-makers

Topics:
{topics_json}

Return ONLY a JSON object: {{"selected_index": <0-9>, "reason": "..."}}
No other text."""

    resp = claude.messages.create(model=MODEL, max_tokens=500,
                                  messages=[{"role": "user", "content": prompt}])
    text = resp.content[0].text.strip()
    m = re.search(r'\{.*\}', text, re.DOTALL)
    result = json.loads(m.group()) if m else json.loads(text)
    idx = int(result['selected_index'])
    return topics[idx], result.get('reason', '')


# ==============================================================
# C: Write the full blog post
# ==============================================================

def write_content(topic, page_url, primary_kw, anchor_text):

    topic_title = topic['Topic Title']
    hook_type = topic['Hook Type']

    prompt = f"""ROLE
You are a senior guest-post and backlink content writer with 10+ years of experience writing for third-party publications across home services, industrial, B2B, and professional industries. You write editorial, informational content for real decision-makers. You do not write marketing copy, sales pages, or SEO filler. Your writing is:
THE BLOG MUST HAVE 1200-1500 words!!! (MANDATORY)
Calm and structured
Factual and grounded in real-world operations
Clear, restrained, and human
Free from hype, buzzwords, or inflated language
DO NOT ADD ANY PICTURES
Add a closing/concluding section
The title should always be H1

INPUTS (MANDATORY)
Target Page URL (for backlink): {page_url}
Primary Keyword: {primary_kw}
Exact Anchor Text: {anchor_text}
Article Title: {topic_title}
Hook Type: {hook_type}

STEP 1: ICP & SEARCH INTENT ANALYSIS (INTERNAL ONLY)
Before writing, infer:
Who the reader is (role and industry context)
Their operational pressure points (risk, downtime, cost, reliability, safety, consistency)
Determine their search intent: Informational, Decision-support, Problem-aware
Adjust accordingly: Tone (calm, explanatory, practical), Language complexity (industry-aware but accessible), Depth (context-first, not tips-first)
Do not mention ICPs, personas, or this analysis in the final output.

STEP 2: INTRODUCTION RULES (STRICT)
The introduction must:
Establish a real-world operational or business context
Explain why the topic matters now, not in theory
Reflect real concerns such as reliability, consistency, or workflow impact
Avoid hooks, hype, or exaggerated framing
Avoid mentioning the backlink, URL, or company
Maintain a neutral, editorial tone
Important: Do not place the anchor text in the introduction. The anchor text must appear only in the first H2 section, close to the top of the article.

STEP 3: STRUCTURE RULES (STRICT)
H2 SECTIONS
Each H2 must represent a major concept, not a tip.
Under every H2, the opening paragraph must:
Explain what the concept is, not what the section "will cover"
Describe the concept directly and concretely
Be written as real informational content, not meta commentary
Do NOT write overview text like "This section explains why..." or "In this section, we discuss..."
Instead, write naturally with direct explanations.
Additional H2 rules: Never jump straight into bullets. Only one anchor text, placed in the first H2 only. Anchor text must feel editorial and supportive of the explanation.

H3 SECTIONS
Each H3 must sit under a relevant H2, expand meaningfully on the H2 concept, explain reasoning, implications, and real-world impact, and connect decisions to outcomes or operational risk.
Avoid one-line explanations, label-style subheadings, and repetition of the H2 idea.

BULLETS
Use bullets only to list applications, summarize outcomes, or reinforce clarity.
Each bullet must be complete, descriptive, and add new information.

STEP 4: CONTENT QUALITY RULES
Build context before advice. Clearly explain cause-and-effect. Emphasize reliability, consistency, and risk reduction. Avoid specifications, measurements, or numerical data. Avoid repetitive structure across sections. Read like an industry explainer, not an SEO article. Express one idea at a time.

STEP 5: BACKLINK INSERTION RULES (NON-NEGOTIABLE)
Insert ONLY ONE backlink.
Use the exact Anchor Text: {anchor_text}
Use the Target Page URL: {page_url}
Placement: Within the first H2 section, early in the article.
Link must feel editorial and support explanation or credibility.
Do NOT add CTAs around the link, mention the company explicitly, or repeat the link.
Primary keyword usage: Use "{primary_kw}" up to 6 times total. Modify slightly if needed for grammatical flow. Do not bold the keyword.

STEP 6: EXTERNAL REFERENCE LINK RULES (STRICT)
Include one authoritative external reference from a site with Domain Rating >= 80 (e.g., Wikipedia, government, standards bodies).
It must be embedded naturally within a sentence. Do not label it as a reference or source. MUST ADD an external link.

STEP 7: LANGUAGE CONSTRAINTS (STRICT)
Do not use: Navigate, Facilitate, Landscape, Tapestry, Leverage, Prowess, Foster, Odyssey, Unlock, Decode, Delve, Unravel, Demystify.
Also avoid buzzwords, sales/marketing language, decorative vocabulary.
Use simple, direct sentences. Clear transitions. Neutral, professional tone.

STEP 8: FINAL QUALITY CHECK
Confirm: Every H2 stands alone with real context. Every H3 adds substance. The backlink feels natural and earned. Keyword usage is organic. The article reads like a legitimate industry guest post.

Write the article now. Output the full article in clean HTML format (h1, h2, h3, p, ul, li, a tags). Do not include any preamble or explanation — only the article HTML."""

    resp = claude.messages.create(model=MODEL, max_tokens=8000,
                                  messages=[{"role": "user", "content": prompt}])
    return resp.content[0].text.strip()


print('✓ LLM pipeline functions defined')

✓ LLM pipeline functions defined


In [ ]:
# ==============================================================
# D: Create Google Doc from HTML  (FIXED VERSION)
# ==============================================================
#
# Fixes applied:
#   1. Newline is inserted BEFORE paragraph style is applied,
#      so heading styles don't bleed into following paragraphs.
#   2. <p> and <li> tags explicitly get NORMAL_TEXT style,
#      preventing inheritance from the previous heading.
#   3. Inline tags (<strong>, <em>, <b>, <i>) are tracked
#      separately so they don't clobber the block-level tag context.
#   4. Bold/italic styling is applied for <strong>/<b> and <em>/<i>.
# ==============================================================

from html.parser import HTMLParser

class HTMLToDocsParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.requests = []
        self.current_index = 1
        self.block_tag = None          # tracks h1, h2, h3, p, li
        self.in_list = False
        self.link_url = None
        self.bold = False
        self.italic = False
        self.pending_segments = []

    def _flush_text(self):
        if not self.pending_segments:
            return

        block_start = self.current_index

        # --- Insert all text segments ---
        for seg in self.pending_segments:
            text = seg['text']
            if not text:
                continue
            start = self.current_index
            end = start + len(text)

            self.requests.append({
                'insertText': {
                    'location': {'index': self.current_index},
                    'text': text
                }
            })
            self.current_index = end

            # Apply link styling
            if seg.get('url'):
                self.requests.append({
                    'updateTextStyle': {
                        'range': {'startIndex': start, 'endIndex': end},
                        'textStyle': {'link': {'url': seg['url']}},
                        'fields': 'link'
                    }
                })

            # Apply bold / italic
            if seg.get('bold') or seg.get('italic'):
                style = {}
                fields = []
                if seg.get('bold'):
                    style['bold'] = True
                    fields.append('bold')
                if seg.get('italic'):
                    style['italic'] = True
                    fields.append('italic')
                self.requests.append({
                    'updateTextStyle': {
                        'range': {'startIndex': start, 'endIndex': end},
                        'textStyle': style,
                        'fields': ','.join(fields)
                    }
                })

        # --- Insert paragraph-ending newline FIRST ---
        self.requests.append({
            'insertText': {
                'location': {'index': self.current_index},
                'text': '\n'
            }
        })
        self.current_index += 1

        # --- NOW apply paragraph style (after newline exists) ---
        tag = self.block_tag
        style_map = {
            'h1': 'HEADING_1',
            'h2': 'HEADING_2',
            'h3': 'HEADING_3',
            'p':  'NORMAL_TEXT',
            'li': 'NORMAL_TEXT',
        }

        if tag in style_map:
            self.requests.append({
                'updateParagraphStyle': {
                    'range': {
                        'startIndex': block_start,
                        'endIndex': self.current_index   # includes the \n
                    },
                    'paragraphStyle': {
                        'namedStyleType': style_map[tag]
                    },
                    'fields': 'namedStyleType'
                }
            })

        self.pending_segments = []

    def handle_starttag(self, tag, attrs):
        ad = dict(attrs)

        # Block-level tags
        if tag in ('h1', 'h2', 'h3', 'p', 'li'):
            self._flush_text()
            self.block_tag = tag
            if tag == 'li':
                self.pending_segments.append({
                    'text': '• ', 'url': None,
                    'bold': False, 'italic': False
                })

        elif tag == 'ul':
            self.in_list = True

        # Inline tags — don't touch block_tag
        elif tag == 'a':
            self.link_url = ad.get('href', '')
        elif tag in ('strong', 'b'):
            self.bold = True
        elif tag in ('em', 'i'):
            self.italic = True

    def handle_endtag(self, tag):
        if tag in ('h1', 'h2', 'h3', 'p', 'li'):
            self._flush_text()
            self.block_tag = None
        elif tag == 'ul':
            self.in_list = False
        elif tag == 'a':
            self.link_url = None
        elif tag in ('strong', 'b'):
            self.bold = False
        elif tag in ('em', 'i'):
            self.italic = False

    def handle_data(self, data):
        if not data.strip() and not self.pending_segments:
            return
        self.pending_segments.append({
            'text': data,
            'url': self.link_url,
            'bold': self.bold,
            'italic': self.italic
        })

# Updated create_doc function  ---> Font and line spacing
def create_google_doc(title, html_content, domain):
    html_content = re.sub(r'^```html\s*', '', html_content)
    html_content = re.sub(r'\s*```$', '', html_content)

    doc = docs_service.documents().create(
        body={'title': title}
    ).execute()
    doc_id = doc['documentId']

    parser = HTMLToDocsParser()
    parser.feed(html_content)
    parser._flush_text()

    if parser.requests:
        docs_service.documents().batchUpdate(
            documentId=doc_id,
            body={'requests': parser.requests}
        ).execute()

    # ─── Apply font and line spacing to entire document ───────────────────────
    doc_content = docs_service.documents().get(documentId=doc_id).execute()
    body_content = doc_content.get('body', {}).get('content', [])

    # Find the full range of the document
    end_index = body_content[-1].get('endIndex', 1) - 1

    style_requests = [
        # Font: Proxima Nova
        {
            'updateTextStyle': {
                'range': {'startIndex': 1, 'endIndex': end_index},
                'textStyle': {
                    'weightedFontFamily': {
                        'fontFamily': 'Proxima Nova'
                    }
                },
                'fields': 'weightedFontFamily'
            }
        },
        # Line spacing: 1.5
        {
            'updateParagraphStyle': {
                'range': {'startIndex': 1, 'endIndex': end_index},
                'paragraphStyle': {
                    'lineSpacing': 150  # 150 = 1.5x in Google Docs API
                },
                'fields': 'lineSpacing'
            }
        }
    ]

    docs_service.documents().batchUpdate(
        documentId=doc_id,
        body={'requests': style_requests}
    ).execute()
    # ──────────────────────────────────────────────────────────────────────────

    drive_service.permissions().create(
        fileId=doc_id,
        body={'type': 'anyone', 'role': 'writer'},
        fields='id'
    ).execute()

    return doc_id, f'https://docs.google.com/document/d/{doc_id}/edit'

# ==============================================================
# E: Log to tracker sheet  (unchanged)
# ==============================================================

def log_to_tracker(wo_key, domain, page, keyword, anchor, bl_website,
                    topic, hook_type, word_count, doc_url, status):
    row = [
        wo_key, domain, page, keyword, anchor,
        bl_website, topic, hook_type, word_count,
        doc_url, status, datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    ]
    tracker_ws.append_row(row, value_input_option='RAW')


print('✓ Google Doc + tracker functions defined')

✓ Google Doc + tracker functions defined


#Step 4

In [ ]:
# ─── TEST MODE CONFIG ─────────────────────────────────────────────────────────
TEST_MODE = False   # Set to False when ready for full production run
TEST_LIMIT = 2     # Number of WOs to process in test mode
# ─────────────────────────────────────────────────────────────────────────────

total = len(pending_df)
success_count = len(completed_keys)
error_count = 0

# Apply test limit if enabled
run_df = pending_df.head(TEST_LIMIT) if TEST_MODE else pending_df

if total == 0:
    print('🎉 Nothing to process — all work orders are already complete!')
else:
    if TEST_MODE:
        print(f'🧪 TEST MODE — processing {TEST_LIMIT} of {total} remaining work orders')
    else:
        print(f'🚀 Processing {total} remaining work orders...')

    print(f'   ({len(completed_keys)} already done, skipped)')
    print(f'📊 Live tracker: {tracker_url}')
    print('=' * 70)

    for idx, row in run_df.iterrows():
        domain = row['Domain']
        page_url = row['Page']
        primary_kw = row['Keyword']
        anchor_text = row['Anchor Text']
        bl_website = row['BL Website']
        wo_key = row['wo_key']

        print(f'\n[{idx + 1}/{len(run_df)}] {domain} → {bl_website}')
        print(f'  KW: {primary_kw} | Anchor: {anchor_text}')

        try:
            # A: Topics
            print(f'  📋 Generating topics...')
            topics = generate_topics(domain, page_url, primary_kw, anchor_text, bl_website)
            print(f'     ✓ {len(topics)} topics')
            time.sleep(API_DELAY)

            # B: Select
            print(f'  🎯 Selecting best topic...')
            selected_topic, reason = select_best_topic(topics, page_url, primary_kw, anchor_text, bl_website)
            print(f'     ✓ "{selected_topic["Topic Title"]}"')
            time.sleep(API_DELAY)

            # C: Write
            print(f'  ✍️  Writing content...')
            html_content = write_content(selected_topic, page_url, primary_kw, anchor_text)
            word_count = len(re.sub(r'<[^>]+>', '', html_content).split())
            print(f'     ✓ ~{word_count} words')
            time.sleep(API_DELAY)

            # D: Google Doc
            print(f'  📄 Creating Google Doc...')
            doc_id, doc_url = create_google_doc(selected_topic['Topic Title'], html_content, domain)
            print(f'     ✓ {doc_url}')

            # E: Log to tracker sheet
            log_to_tracker(
                wo_key, domain, page_url, primary_kw, anchor_text,
                bl_website, selected_topic['Topic Title'],
                selected_topic['Hook Type'], word_count, doc_url, 'Success'
            )

            # F: Write doc URL back to WO sheet
            wo_ws.update_cell(row['_sheet_row'], content_link_col, doc_url)
            wo_ws.update_cell(row['_sheet_row'], status_col, 'Blog Generated')
            print(f'  📝 WO sheet updated → row {row["_sheet_row"]}')

            success_count += 1
            print(f'  ✅ Done [{success_count} total succeeded]')

        except Exception as e:
            print(f'  ❌ ERROR: {e}')
            error_count += 1
            try:
                log_to_tracker(
                    wo_key, domain, page_url, primary_kw, anchor_text,
                    bl_website, '', '', 0, '', f'Error: {str(e)[:200]}'
                )
            except:
                print(f'  ⚠ Could not log error to sheet')

        print('-' * 70)

    print(f'\n{"=" * 70}')
    if TEST_MODE:
        print(f'🧪 Test run complete! Check the docs above.')
        print(f'   If happy, set TEST_MODE = False and re-run for all {total} remaining WOs')
    else:
        print(f'✅ Pipeline complete!')
    print(f'   Succeeded: {success_count} | Failed: {error_count}')
    print(f'📊 Full results: {tracker_url}')

🚀 Processing 7 remaining work orders...
   (585 already done, skipped)
📊 Live tracker: https://docs.google.com/spreadsheets/d/1z_daQxBo7WAQNRLCp8CAWJz7ZzaeHbZnWbnlpJgQk3M/edit

[1/7] assembly-industries.com → bignewsnetwork.com
  KW: employee onboarding offboarding solutions | Anchor: employee onboarding offboarding solutions
  📋 Generating topics...
     ✓ 10 topics
  🎯 Selecting best topic...
     ✓ "Beyond Paperwork: How Next-Generation Employee Onboarding Offboarding Solutions Are Transforming U.S. Workforce Operations"
  ✍️  Writing content...
     ✓ ~1717 words


  📄 Creating Google Doc...


     ✓ https://docs.google.com/document/d/1Bx27dv6yRDwSgeGy39B74rDfjg-HO4Syqezx7ucHhD4/edit
  📝 WO sheet updated → row 589
  ✅ Done [586 total succeeded]
----------------------------------------------------------------------

[2/7] assembly-industries.com → seterra.io
  KW: employee onboarding offboarding solutions | Anchor: employee onboarding offboarding solutions
  📋 Generating topics...
     ✓ 10 topics
  🎯 Selecting best topic...
     ✓ "5 Signs Your Employee Onboarding Process Is Costing Your Company More Than You Think"
  ✍️  Writing content...
     ✓ ~1801 words
  📄 Creating Google Doc...
     ✓ https://docs.google.com/document/d/1kzoKmPto0h0N0-EMlAnGJc7YhZebuaP69DN6ZYHtQ9U/edit
  📝 WO sheet updated → row 590
  ✅ Done [587 total succeeded]
----------------------------------------------------------------------

[3/7] assembly-industries.com → gracejabbaribio.com
  KW: employee onboarding offboarding solutions | Anchor: employee onboarding offboarding solutions
  📋 Generating top